In [11]:
# import necessary libraries

import os
from dotenv import load_dotenv
load_dotenv()


# Import pypdf loader for langchain
from langchain_community.document_loaders import PyPDFLoader

In [12]:
# Set the api key

api_key = os.getenv("demo_api_key")

In [13]:
# Creating function to load api key

def load_api_key(path):
    if path:
        print("API key loaded successfully.")
        return path
    else:
        raise ValueError("API key not found. Please set the API key in the .env file.") 


In [14]:
# Calling the api key function

api_key = load_api_key(api_key)

API key loaded successfully.


In [15]:
# Load the PDF document

# Method 1: Calling it directly from path

pdf_path = 'Adelegan_Deborah_CV.pdf'

# Using PyPDFLoader to load the PDF
loader = PyPDFLoader(pdf_path)
pages = loader.load()

In [16]:
# Using a function to print pages
for page in pages:
    print(page.page_content)

ADELEGAN DEBORAH JESUDEMILADE  
📍 18, Anuoluwapo Street, Off Purposeful Avenue, Olambe, Ogun State 
📍 +2349079790073 | 📍 deborahjesudemilade@gmail.com  
📍 LinkedIn: https://www.linkedin.com/in/deborah-adelegan/ 
Professional Summary 
Biochemistry graduate with strong interest in tech and growing skills in data analysis. I’ve 
trained with Tech4Dev, and am currently taking a data science course at WorldQuant 
University.  I am eager to grow in tech and contribute to innovative solutions. 
Technical Skills 
Languages/Tools: Python, SQL, Excel, Tableau 
Concepts: Data analysis, data cleaning, data visualization, exploratory data analysis (EDA) 
Platforms: Jupyter Notebook, Google Colab 
Soft Skills: Communication, teamwork, leadership, adaptability, problem-solving 
Work & Internship Experience 
Data Science Fellow 
Tech4Dev – Women Techsters Fellowship | Aug 2022– Feb 2023 
- Completed a 6-month intensive training program in data science and analytics. 
- Gained hands-on experience in Py

In [17]:
# Method 2 - Using a Function

def load_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    return pages

pages = load_pdf(pdf_path)

In [18]:
# Using text splitter 
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Set chunk size and overlap insode splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap = 100)

pdf_documents = splitter.split_documents(documents=pages)      

pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-05-12T12:03:56+00:00', 'author': 'python-docx', 'moddate': '2025-05-12T12:03:57+00:00', 'source': 'Adelegan_Deborah_CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='ADELEGAN DEBORAH JESUDEMILADE  \n📍 18, Anuoluwapo Street, Off Purposeful Avenue, Olambe, Ogun State \n📍 +2349079790073 | 📍 deborahjesudemilade@gmail.com  \n📍 LinkedIn: https://www.linkedin.com/in/deborah-adelegan/ \nProfessional Summary \nBiochemistry graduate with strong interest in tech and growing skills in data analysis. I’ve \ntrained with Tech4Dev, and am currently taking a data science course at WorldQuant \nUniversity.  I am eager to grow in tech and contribute to innovative solutions. \nTechnical Skills \nLanguages/Tools: Python, SQL, Excel, Tableau \nConcepts: Data analysis, data cleaning, data visualization, exploratory data analysis (EDA) \nPlatforms: Jupyter Notebook, Google Colab 

# **Embedding**

In [19]:
from langchain_openai import OpenAIEmbeddings

In [20]:
embeddings = OpenAIEmbeddings(
    model = 'text-embedding-3-small', 
    openai_api_key = api_key,
)

embedding_testing = embeddings.embed_query('What is Pharmacovigilance?')
print('Embedding Shape:', len(embedding_testing))
print('In ascending order', embedding_testing.sort())

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# **Vector Store**

In [ ]:
from langchain_community.vectorstores import FAISS

vectorbank = FAISS.from_documents(pdf_documents, embeddings)

print(f'The vector store has been created and it contains {len(pdf_documents)} chunks.')

# Create a query and check for the similarity
query =  'What are the skills a data analyst needs?'
results = vectorbank.similarity_search(query, k=2)

print(f'Query: {query}')
print(results)


# **Application**

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Create the model
llm = ChatOpenAI(
    model = 'gpt-3.5-turbo',
    temperature=0,
    openai_api_key=api_key
)

# Set the persona
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant to review the documents supplied and answer questions related to tech skills. Answer using ONLY the provided context. Do not provide information not supplied in the document."),
    ("human", "Question: {question}\n\n:\n{context}")
])  

# Apply the retrieval parameters
text_retriever = vectorbank.as_retriever(
    search_kwargs={'k':4}
)

# Create a function to return the result
def document_formatter(doc):
    return "\n\n".join(item.page_content for item in doc)

In [ ]:
#
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context":text_retriever | document_formatter,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)

In [ ]:
response = rag_chain.invoke('What are the skills a data analyst needs?')
print(response.content)